# 09. Attention Mechanism Intro Practice

这一练习覆盖 causal mask、masked softmax 和 scaled dot-product attention。

In [ ]:
import math
import torch


In [ ]:
def build_causal_mask(seq_len):
    return torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool))


def masked_softmax(scores, mask=None, dim=-1):
    if mask is not None:
        scores = scores.masked_fill(~mask, float('-inf'))
    shifted = scores - scores.max(dim=dim, keepdim=True).values
    probs = torch.softmax(shifted, dim=dim)
    if mask is not None:
        probs = probs * mask.to(dtype=probs.dtype)
        probs = probs / probs.sum(dim=dim, keepdim=True).clamp_min(1e-12)
    return probs


def attention_weights(q, k, mask=None):
    d_k = q.shape[-1]
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    return masked_softmax(scores, mask=mask, dim=-1)


def scaled_dot_product_attention(q, k, v, mask=None):
    weights = attention_weights(q, k, mask=mask)
    output = torch.matmul(weights, v)
    return output, weights


In [ ]:
def test_build_causal_mask():
    mask = build_causal_mask(4)
    expected = torch.tensor([
        [True, False, False, False],
        [True, True, False, False],
        [True, True, True, False],
        [True, True, True, True],
    ])
    assert torch.equal(mask, expected)
    print('✅ build_causal_mask 通过')


def test_masked_softmax():
    scores = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
    mask = torch.tensor([[True, True, False], [True, True, True]])
    probs = masked_softmax(scores, mask=mask)
    assert torch.allclose(probs.sum(dim=-1), torch.ones(2), atol=1e-6, rtol=1e-6)
    assert torch.allclose(probs[0, 2], torch.tensor(0.0))
    print('✅ masked_softmax 通过')


def test_attention_weights():
    q = torch.tensor([[[1.0, 0.0], [0.0, 1.0]]])
    k = torch.tensor([[[1.0, 2.0], [3.0, 4.0]]])
    mask = build_causal_mask(2).unsqueeze(0)
    weights = attention_weights(q, k, mask=mask)
    expected = torch.tensor([[[1.0, 0.0], [0.19557033, 0.80442965]]])
    assert torch.allclose(weights, expected, atol=1e-5, rtol=1e-5)
    print('✅ attention_weights 通过')


def test_scaled_dot_product_attention():
    q = torch.tensor([[[1.0, 0.0], [0.0, 1.0]]])
    k = torch.tensor([[[1.0, 2.0], [3.0, 4.0]]])
    v = torch.tensor([[[10.0, 0.0], [0.0, 20.0]]])
    mask = build_causal_mask(2).unsqueeze(0)
    output, weights = scaled_dot_product_attention(q, k, v, mask=mask)
    expected_weights = attention_weights(q, k, mask=mask)
    assert torch.allclose(weights, expected_weights, atol=1e-6, rtol=1e-6)
    assert output.shape == (1, 2, 2)
    print('✅ scaled_dot_product_attention 通过')


test_build_causal_mask()
test_masked_softmax()
test_attention_weights()
test_scaled_dot_product_attention()


In [ ]:
q = torch.tensor([[[1.0, 0.0, 1.0], [0.0, 1.0, 1.0]]])
k = torch.tensor([[[1.0, 2.0, 0.0], [3.0, 0.0, 1.0]]])
v = torch.tensor([[[10.0, 0.0], [0.0, 20.0]]])
mask = build_causal_mask(2).unsqueeze(0)
output, weights = scaled_dot_product_attention(q, k, v, mask=mask)
print('weights:')
print(weights)
print('output:')
print(output)
